# 04   Graph Construction

**Pipeline position:** `01 -> 02 -> 03a -> 03b -> **04**`.

**Purpose:** build the deterministic **knowledge graph** over the corpus that the GraphRAG retrieval layer (05) expands over. Nodes and edges are joined on the 03b **lineage IDs**, so any retrieved chunk lands directly on the graph.

Node kinds: `document`, `preamble`, `article`, `external_article`, `external_document`, `term`, `entity`.

Edge kinds (each carries `evidence` = `{doc_id, offset, snippet≤120}`; `unresolved: true` for external destinations):
- `CROSS_REFERENCES`   article -> article. From `Article N` mentions in the prose (tables excluded). Lists/ranges (`Articles 2, 3 and 4`) expand into one edge per member. Instrument context after the mention (`of Regulation (EU) 2016/679`) resolves the target across documents inside the corpus; otherwise an `ext:article` stub.
- `AMENDS`   hosting node -> instrument document. (`amended by Regulation (EU) 2024/1106`). Out-of-corpus instruments get an `ext:instrument` stub (provenance preserved).
- `DEFINED_IN`   term node -> the article/preamble that hosts the (`'term' means ...`) definition.
- `APPLIES_TO`   hosting node -> scope entity (closed-vocab heuristic, `confidence: 'heuristic'`).

Engine: `src/graph_builder.py`   deterministic, regex only, no LLM, idempotent.

Outputs (under `notebooks/data/graph/`): `nodes.jsonl`, `edges.jsonl`, `graph_summary.csv`, `graph_handoff.json`.

---
## 0   Setup

In [1]:
import os, sys, json, time
from pathlib import Path

import pandas as pd
from rich.console import Console
from rich import print as rprint

Console()

def _repo_root() -> Path:
    env = os.getenv("ENERGY_AUDIT_ROOT")
    if env and Path(env).exists():
        return Path(env).resolve()
    cur = Path.cwd().resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "notebooks").is_dir() and (cand / "data").is_dir():
            return cand
    return cur.parent

ROOT = _repo_root()
sys.path.insert(0, str(ROOT / "src"))

import common as c
from structure import graph_builder as GB
import json as _json

GRAPH_DIR = c.NOTEBOOKS_DATA / "graph"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
rprint("[bold]root :[/bold]", "../" + ROOT.name)
rprint("[bold]outputs :[/bold]", GRAPH_DIR.relative_to(ROOT))


root  : <repo root>

outputs : notebooks/data/graph

---
## 1   Load corpus, build and persist the graph

In [2]:
t0 = time.time()
recs = GB.load_corpus()
rprint("[bold]loaded[/bold]", len(recs), "docs |", sum(len(r.art_titles) for r in recs), "structural articles")

result = GB.build_graph(recs)
saved = GB.save_graph(result, GRAPH_DIR)

s = result["summary"]
rprint("[bold]built in %.1fs[/bold]" % (time.time() - t0))
rprint("nodes:", s["nodes"], "| edges:", s["edges"])
rprint("kinds:", s["kinds"])
rprint("unresolved (external destinations):", s["unresolved"])


loaded 25 docs | 1138 structural articles

built in 0.8s

nodes: 2049 | edges: 4139

kinds:
{'CROSS_REFERENCES': 3186, 'AMENDS': 54, 'DEFINED_IN': 839, 'APPLIES_TO': 60}

unresolved (external destinations): 721

---
## 2   Validate

All must hold:
- no duplicate `(src, dst, kind)` edges
- every edge endpoint resolves to a node (no dangling lineages)
- every edge carries `doc_id`, `offset`, `snippet` (evidence)
- snippet length ≤ 120 chars
- per-kind counts are non-negative and sum to total
- `unresolved` flag present on every edge

In [3]:

nodes = [_json.loads(l) for l in (GRAPH_DIR / "nodes.jsonl").read_text().splitlines() if l.strip()]
edges = [_json.loads(l) for l in (GRAPH_DIR / "edges.jsonl").read_text().splitlines() if l.strip()]
node_ids = {n["lineage_id"] for n in nodes}

problems = []
if len({(e["src"], e["dst"], e["kind"]) for e in edges}) != len(edges):
    problems.append("duplicate (src,dst,kind) edges")
dangling = [e for e in edges if e["src"] not in node_ids or e["dst"] not in node_ids]
if dangling:
    problems.append("%d dangling endpoints, e.g. %s -> %s" % (len(dangling), dangling[0]["src"], dangling[0]["dst"]))
noev = [e for e in edges if not ("doc_id" in e and "offset" in e and "snippet" in e)]
if noev: problems.append("%d edges missing evidence" % len(noev))
long = [e for e in edges if len(e.get("snippet","")) > 120]
if long: problems.append("%d snippets > 120 chars" % len(long))
if sum(s["kinds"].values()) != s["edges"]: problems.append("kind counts != total")
if any("unresolved" not in e for e in edges): problems.append("missing unresolved flag")

if problems:
    rprint("[bold red]VALIDATION FAILURES:[/bold red]")
    for p in problems: rprint("  ", p)
    raise AssertionError("%d graph problems" % len(problems))

rprint("[bold green]graph validates clean:[/bold green]")
rprint("   no duplicate edges  |  %d edges, %d nodes" % (len(edges), len(nodes)))
rprint("   no dangling endpoints  |  evidence (doc_id, offset, snippet) on 100% of edges")
rprint("   all snippets ≤120 chars  |  unresolved flag on every edge")


graph validates clean:

no duplicate edges  |  4139 edges, 2049 nodes

no dangling endpoints  |  evidence (doc_id, offset, snippet) on 100% of edges

all snippets ≤120 chars  |  unresolved flag on every edge

---
## 3   Per-doc summary + samples

In [4]:
df = pd.read_csv(GRAPH_DIR / "graph_summary.csv")
df = df.sort_values(["cross", "defined_in", "amends", "applies_to"], ascending=False)
rprint("[bold]per-doc edge counts (top 8 by cross-refs):[/bold]")
rprint(df.head(8).to_string(index=False))

def _show(kind, unresolved=None, n=4):
    es = [e for e in edges if e["kind"] == kind]
    if unresolved is not None:
        es = [e for e in es if e["unresolved"] == unresolved]
    rprint("\n[bold]%s = %d[/bold]" % (kind, len(es)))
    for e in es[:n]:
        rprint("   %s -> %s" % (e["src"], e["dst"]))
        rprint("      | %s" % e["snippet"][:88])

_show("CROSS_REFERENCES", unresolved=False)
_show("CROSS_REFERENCES", unresolved=True, n=2)
_show("AMENDS", n=3)
_show("DEFINED_IN", n=3)
_show("APPLIES_TO", n=3)

per-doc edge counts (top 8 by cross-refs):

doc_id  cross  amends  defined_in  applies_to  unresolved
entso_sogl_2017_1485    493       0         155           0          33
      mica_2023_1114    266       5          47           6         110
       gdpr_2016_679    263       0          25           2           2
entso_ebgl_2017_2195    237       0          45           3          22
   elec_reg_2019_943    225       1          71           1          32
entso_cacm_2015_1222    210       0          45           1           2
   elec_dir_2019_944    202       1          61           4          23
      dora_2022_2554    196       5          59           9          78

CROSS_REFERENCES = 2510

data_act_2023_2854:article:1 -> gdpr_2016_679:article:15

| Articles 15 and 20 of Regulation (EU) 2016/679. In the event of a conflict between this

...

| applies to market participants who possess inside information in respect of business or

acer_remit_guidance:document -> corpus:entity:entities

| applies to the groups of entities specified in Article 8(4) and Article 8(5) of REMIT, t

acer_remit_guidance:document -> corpus:entity:transmission_system_operators

| apply to transmission system operators when purchasing electricity or natural gas in ord